In [155]:
import pandas as pd
from Bio import Entrez
import os


This notebook will take an existing network and add mesh term keywords as nodes to the existing networks. Additionally this will add extraneous information like title, its publisher, etc.

In [156]:
path_to_folder = "../CFDE/"

Here we will gather all of the pmids across the entire network

In [157]:
pmids = pd.read_csv(path_to_folder + "pmids.nodes.csv")
pmids_list = list(pmids['label'])


In [160]:
pmids

,id,label,url
0,1969,38191932,https://pubmed.ncbi.nlm.nih.gov/38191932/
1,1970,37452018,https://pubmed.ncbi.nlm.nih.gov/37452018/
2,1971,30865299,https://pubmed.ncbi.nlm.nih.gov/30865299/
3,1972,39420002,https://pubmed.ncbi.nlm.nih.gov/39420002/
4,1973,37090499,https://pubmed.ncbi.nlm.nih.gov/37090499/
...,...,...,...
185,2154,35833142,https://pubmed.ncbi.nlm.nih.gov/35833142/
186,2155,38413840,https://pubmed.ncbi.nlm.nih.gov/38413840/
187,2156,39799122,https://pubmed.ncbi.nlm.nih.gov/39799122/
188,2157,36001024,https://pubmed.ncbi.nlm.nih.gov/36001024/


In [162]:
def fetch_article_details(pmid_list):
    handle = Entrez.efetch(db="pubmed", id=pmid_list, retmode="xml")
    records = Entrez.read(handle)
    handle.close()

    articles = []
    for record in records['PubmedArticle']:
        article = {}
        medline_citation = record['MedlineCitation']
        article_data = medline_citation.get('Article', {})

        # Basic info
        article['PMID'] = medline_citation['PMID']
        article['Title'] = article_data.get('ArticleTitle', 'N/A')
        journal_info = article_data.get('Journal', {})

        # Journal details
        article['Journal'] = journal_info.get('Title', 'N/A')
        article['Volume'] = journal_info.get('JournalIssue', {}).get('Volume', 'N/A')
        article['Issue'] = journal_info.get('JournalIssue', {}).get('Issue', 'N/A')
        article['Pages'] = article_data.get('Pagination', {}).get('MedlinePgn', 'N/A')
        
        # Extract publication year
        pub_date = journal_info.get('JournalIssue', {}).get('PubDate', {})
        article['Year'] = pub_date.get('Year', pub_date.get('MedlineDate', 'N/A'))  # Handle cases where Year is missing

        # MeSH terms
        mesh_terms = medline_citation.get('MeshHeadingList', [])
        article['MeSH Terms'] = [mesh['DescriptorName'] for mesh in mesh_terms]

        articles.append(article)
    return articles


In [163]:
articles = fetch_article_details(",".join(map(str, pmids_list)))
df = pd.DataFrame(articles)
print(df)

/Users/MaayanLab/miniconda3/envs/maayanlab/lib/python3.13/site-packages/Bio/Entrez/__init__.py:734: UserWarning: 
            Email address is not specified.

            To make use of NCBI's E-utilities, NCBI requires you to specify your
            email address with each request.  As an example, if your email address
            is A.N.Other@example.com, you can specify it as follows:
               from Bio import Entrez
               Entrez.email = 'A.N.Other@example.com'
            In case of excessive usage of the E-utilities, NCBI will attempt to contact
            a user at the email address provided before blocking access to the
            E-utilities.
  warnings.warn(


         PMID                                              Title  \
0    38191932  A fast, scalable and versatile tool for analys...   
1    37452018  Lactate-dependent transcriptional regulation c...   
2    30865299  Ancestral characterization of 1018 cancer cell...   
3    39420002  Methionine-SAM metabolism-dependent ubiquinone...   
4    37090499  Evaluating cancer cell line and patient-derive...   
..        ...                                                ...   
185  35833142  Construction and Application of Polygenic Risk...   
186  38413840  Enhancer selectivity in space and time: from e...   
187  39799122  Homo Sapiens Chromosomal Location Ontology: A ...   
188  36001024  Proteogenomic Markers of Chemotherapy Resistan...   
189  37242535  Network Biology-Inspired Machine Learning Feat...   

                                       Journal Volume Issue      Pages  Year  \
0                               Nature methods     21     2    217-227  2024   
1                      

Let us first add title and journal fields to the pmid nodes.

In [164]:
pmid_to_title_journal = {row['PMID']: (row['Title'], row['Journal'], row['Volume'], row['Issue'], row['Pages'], row['Year']) for _, row in df.iterrows()}
mapped_df = pd.DataFrame.from_dict(pmid_to_title_journal, orient='index', columns=['Title', 'Journal', 'Volume', 'Issue', 'Pages', 'Year'])
mapped_df.index.name = 'label'
mapped_df.index = mapped_df.index.astype(str)

In [165]:
pmids["label"] = pmids["label"].astype(str)
pmids = pmids.merge(mapped_df, left_on='label', right_index=True, how='left')


In [166]:
pmids.to_csv(path_to_folder + "pmids.nodes.csv", index=False)

Now let us create MeSH Term nodes and then edges to relevant PMIDS and authors

In [167]:
pmid_to_mesh = {row['PMID']: (row['MeSH Terms']) for _, row in df.iterrows()}
unique_mesh_terms = set()

for mesh_terms in pmid_to_mesh.values():
    if isinstance(mesh_terms, list):  # Ensure it's iterable
        unique_mesh_terms.update(mesh_terms)
    else:  # Handle single terms stored incorrectly
        unique_mesh_terms.add(mesh_terms)

unique_mesh_terms = list(unique_mesh_terms)


# Let us get the highest id in all of the nodes.
max_idx = 0
for file in os.listdir(path_to_folder):
    if "nodes" in file:
        temp_df = pd.read_csv(path_to_folder + file)
        temp_max = temp_df['id'].max()
        if max_idx < temp_max:
            max_idx = temp_max
        else:
            pass
        del temp_df
        del temp_max

mesh_nodes = pd.DataFrame()
id_list = [i for i in range(max_idx, max_idx + len(unique_mesh_terms))]
mesh_nodes['id'] = id_list
del id_list
mesh_nodes['label'] = unique_mesh_terms
mesh_nodes.to_csv(path_to_folder + "mesh.nodes.csv", index=False)

In [168]:
# edges
# we want edges to and from mesh terms to Pubmed Labels.
map_pmid_id = dict(zip(pmids["label"], pmids["id"]))
map_mesh_id = dict(zip(mesh_nodes["label"], mesh_nodes["id"]))

sources = []
targets = []
edge_name = "MeSH Term"
for key, val in pmid_to_mesh.items():
    for tar in val:
        sources.append(map_pmid_id[key])
        targets.append(map_mesh_id[tar])

Mesh_edges = pd.DataFrame()
Mesh_edges['source'] = sources
Mesh_edges['relation'] = [edge_name] * len(sources)
Mesh_edges['target'] = targets

Mesh_edges.to_csv(path_to_folder + "pmids.MeSH Term.mesh.edges.csv", index=False)

Mesh_edges['source'] = targets
Mesh_edges['target'] = sources
Mesh_edges.to_csv(path_to_folder + "mesh.MeSH Term.pmids.edges.csv", index=False)

In [169]:
pmids

,id,label,url,Title,Journal,Volume,Issue,Pages,Year
0,1969,38191932,https://pubmed.ncbi.nlm.nih.gov/38191932/,"A fast, scalable and versatile tool for analys...",Nature methods,21,2,217-227,2024
1,1970,37452018,https://pubmed.ncbi.nlm.nih.gov/37452018/,Lactate-dependent transcriptional regulation c...,Nature communications,14,1,4129,2023
2,1971,30865299,https://pubmed.ncbi.nlm.nih.gov/30865299/,Ancestral characterization of 1018 cancer cell...,Cancer,125,12,2076-2088,2019
3,1972,39420002,https://pubmed.ncbi.nlm.nih.gov/39420002/,Methionine-SAM metabolism-dependent ubiquinone...,Nature communications,15,1,8971,2024
4,1973,37090499,https://pubmed.ncbi.nlm.nih.gov/37090499/,Evaluating cancer cell line and patient-derive...,bioRxiv : the preprint server for biology,N/A,N/A,N/A,2023
...,...,...,...,...,...,...,...,...,...
185,2154,35833142,https://pubmed.ncbi.nlm.nih.gov/35833142/,Construction and Application of Polygenic Risk...,Frontiers in immunology,13,N/A,889296,2022
186,2155,38413840,https://pubmed.ncbi.nlm.nih.gov/38413840/,Enhancer selectivity in space and time: from e...,Nature reviews. Molecular cell biology,25,7,574-591,2024
187,2156,39799122,https://pubmed.ncbi.nlm.nih.gov/39799122/,Homo Sapiens Chromosomal Location Ontology: A ...,Scientific data,12,1,52,2025
188,2157,36001024,https://pubmed.ncbi.nlm.nih.gov/36001024/,Proteogenomic Markers of Chemotherapy Resistan...,Cancer discovery,12,11,2586-2605,2022


In [176]:
# Swap each label with title abbreviated
pmid = pmids['label']
pmid_titles = pmids['Title'].str[:40] + "..."
pmids['label'] = pmid_titles
pmids['PMID'] = pmids_list

pmids.to_csv(path_to_folder + "pmids.nodes.csv", index=False)

In [177]:
pmids

,id,label,url,Title,Journal,Volume,Issue,Pages,Year,PMID
0,1969,"A fast, scalable and versatile tool for ...",https://pubmed.ncbi.nlm.nih.gov/38191932/,"A fast, scalable and versatile tool for analys...",Nature methods,21,2,217-227,2024,38191932
1,1970,Lactate-dependent transcriptional regula...,https://pubmed.ncbi.nlm.nih.gov/37452018/,Lactate-dependent transcriptional regulation c...,Nature communications,14,1,4129,2023,37452018
2,1971,Ancestral characterization of 1018 cance...,https://pubmed.ncbi.nlm.nih.gov/30865299/,Ancestral characterization of 1018 cancer cell...,Cancer,125,12,2076-2088,2019,30865299
3,1972,Methionine-SAM metabolism-dependent ubiq...,https://pubmed.ncbi.nlm.nih.gov/39420002/,Methionine-SAM metabolism-dependent ubiquinone...,Nature communications,15,1,8971,2024,39420002
4,1973,Evaluating cancer cell line and patient-...,https://pubmed.ncbi.nlm.nih.gov/37090499/,Evaluating cancer cell line and patient-derive...,bioRxiv : the preprint server for biology,N/A,N/A,N/A,2023,37090499
...,...,...,...,...,...,...,...,...,...,...
185,2154,Construction and Application of Polygeni...,https://pubmed.ncbi.nlm.nih.gov/35833142/,Construction and Application of Polygenic Risk...,Frontiers in immunology,13,N/A,889296,2022,35833142
186,2155,Enhancer selectivity in space and time: ...,https://pubmed.ncbi.nlm.nih.gov/38413840/,Enhancer selectivity in space and time: from e...,Nature reviews. Molecular cell biology,25,7,574-591,2024,38413840
187,2156,Homo Sapiens Chromosomal Location Ontolo...,https://pubmed.ncbi.nlm.nih.gov/39799122/,Homo Sapiens Chromosomal Location Ontology: A ...,Scientific data,12,1,52,2025,39799122
188,2157,Proteogenomic Markers of Chemotherapy Re...,https://pubmed.ncbi.nlm.nih.gov/36001024/,Proteogenomic Markers of Chemotherapy Resistan...,Cancer discovery,12,11,2586-2605,2022,36001024
